# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityag30/FlyRank-internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For this task, I selected a **Decision Tree Classifier**.

A Decision Tree is appropriate because it learns simple decision rules from the data, making it easy to compare with the hand-written baseline created in Week 4. Unlike more complex models, a Decision Tree provides interpretable splits that explain why a page is classified as a refresh candidate or not.

The model can capture interactions between multiple features, such as impressions, click-through rate (CTR), average position, and engagement rate, while remaining easy to visualize and explain.

The objective is not to maximize model complexity but to determine whether a learned model can outperform the rule-based baseline using the same data and evaluation metric.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Selected modeling method
method = "Decision Tree Classifier"

print("Selected Method:", method)

Selected Method: Decision Tree Classifier


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


An 80/20 train-test split was used with stratified sampling to preserve the proportion of refresh candidates in both sets. The same feature set was used for both the baseline and the Decision Tree model to ensure a fair comparison. Only observable search and engagement signals from March 2026 were used, preventing leakage from future information or product-generated decision flags.

In [5]:
# This cell is for CODE (numbers, a query, a check).
from google.colab import userdata
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split

# -----------------------------
# Connect to warehouse
# -----------------------------
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

# -----------------------------
# Load March 2026 data
# -----------------------------
query = f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY content_hash_id
"""

df = con.sql(query).df()

# -----------------------------
# Missing values
# -----------------------------
numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

df[numeric_cols] = df[numeric_cols].fillna(0)

# -----------------------------
# Feature engineering
# -----------------------------
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, 1)
)

df["engagement_rate"] = (
    df["ga4_engaged_sessions"] /
    df["ga4_sessions"].replace(0, 1)
)

# -----------------------------
# Baseline target
# -----------------------------
df["target"] = (
    (df["gsc_impressions"] >= 10000) &
    (df["ctr"] < 0.03)
).astype(int)

# -----------------------------
# Features
# -----------------------------
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ctr",
    "engagement_rate"
]

X = df[feature_cols]
y = df["target"]

# -----------------------------
# Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))
print("Refresh pages (train):", y_train.sum())
print("Refresh pages (test):", y_test.sum())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training samples: 265149
Testing samples : 66288
Refresh pages (train): 4701
Refresh pages (test): 1175


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


A Decision Tree classifier was trained using the same training and testing split as the baseline. Both methods were evaluated on the same test set using accuracy, precision, recall, and F1-score so the comparison is fair.

In [6]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd


# Train Decision Tree

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
tree_pred = model.predict(X_test)


# Baseline Prediction

baseline_pred = (
    (X_test["gsc_impressions"] >= 10000) &
    (X_test["ctr"] < 0.03)
).astype(int)


# Evaluation

results = pd.DataFrame({
    "Method": ["Baseline Rule", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, tree_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, tree_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, tree_pred, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, tree_pred, zero_division=0)
    ]
})

print(results)

          Method  Accuracy  Precision    Recall  F1 Score
0  Baseline Rule  1.000000        1.0  1.000000  1.000000
1  Decision Tree  0.999985        1.0  0.999149  0.999574


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Decision Tree was inspected by comparing its predictions with the true labels on the test set. Misclassified pages were reviewed to understand common failure patterns. Feature importance was also examined to identify which observable signals contributed most to the model's decisions. This helps explain both the strengths and limitations of the model beyond overall accuracy.

In [7]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.metrics import confusion_matrix
import pandas as pd

# Confusion Matrix

cm = confusion_matrix(y_test, tree_pred)

print("Confusion Matrix")
print(cm)

# Misclassified examples
errors = X_test.copy()
errors["Actual"] = y_test.values
errors["Predicted"] = tree_pred

misclassified = errors[errors["Actual"] != errors["Predicted"]]

print("\nNumber of Misclassified Samples:", len(misclassified))
display(misclassified.head(10))

# Feature Importance
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance")
display(importance)

Confusion Matrix
[[65113     0]
 [    1  1174]]

Number of Misclassified Samples: 1


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ctr,engagement_rate,Actual,Predicted
309233,10001.0,25.0,5.715954,6.0,0.0,0.0025,0.0,1,0



Feature Importance


,Feature,Importance
0,gsc_impressions,0.999783
5,ctr,0.000217
1,gsc_clicks,0.000000
2,gsc_avg_position,0.000000
3,ga4_sessions,0.000000
4,ga4_engaged_sessions,0.000000
6,engagement_rate,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.